### 0. Imports and Variables ###

In [7]:
from pathlib import Path
import json
import os
import pandas as pd

# Colors are defined only here as (R, G, B) tuples
ColorPrimary = (220,120,0)
ColorSecondary = (0,180,220)

# Hex versions for matplotlib/seaborn, derived from the tuples above
color_primary = '#%02X%02X%02X' % ColorPrimary
color_secondary = '#%02X%02X%02X' % ColorSecondary

# output_notebook(resources=INLINE) # --- BOKEH JUPYTERLAB SETUP ---

# Dataframes for calculations
# allDataDF
# filteredDF
# firstMarksDF

### 1. Load JSON data ###

Excluded datasets:

8 participants failed to provide any marks at all or faced technical difficulties. Their records were therefore excluded from further analysis.

Datasets with no valid marks: client0134, client23221, client50666, client178781, client310002, client357578, client375575
Incomplete datasets: improethics.clientundefined


In [8]:
folder_path = Path("../data/processedData/jsonData") # Directory containing JSON files
json_files = list(folder_path.glob("*.json"))

### 2. Regroup data with marks at the top level ###

In [9]:
allDataDF = pd.DataFrame()
data_list = []
df_list = []

for json_file in json_files:
    try:
        with open(json_file, 'r') as file:
            loadedData = json.load(file)
            # Flatten the data if it is a list (the data from the research concert is a list!)
            if isinstance(loadedData, list):
                data_list.extend(loadedData)  # Add all elements from the list
            elif isinstance(loadedData, dict):
                data_list.append(loadedData)  # Add the single dictionary
            else:
                print(f"Unsupported JSON structure in file: {json_file}")
            
            # Extract user information from the config data
            user_configs = [item for item in loadedData if item.get('dataset') == 'config']
            if not user_configs:
                print(f"No config found in {json_file}")
                continue
            user_info = user_configs[0]
            
            user_id = user_info.get('id')
            if isinstance(user_id, dict) and '$numberDouble' in user_id:
                user_id = float(user_id['$numberDouble'])

            user_data = {
                'userID': user_id,
                'age': user_info.get('personalBackground', {}).get('age'),
                'gender': user_info.get('personalBackground', {}).get('gender'),
                'musicaltraining': user_info.get('personalBackground', {}).get('musicaltraining'),
                'experienceimprovising': user_info.get('personalBackground', {}).get('experienceimprovising')
            }
            # Extract timestamps from marks
            marks_datasets = [item for item in loadedData if item.get('dataset') == 'marks']
            if not marks_datasets:
                print(f"No marks found in {json_file}")
                timestamps = []
            else:
                marks = marks_datasets[0].get('marks', [])
                timestamps = [mark.get('timeStamp') for mark in marks if 'timeStamp' in mark]
    
            # Create DataFrame with timestamps
            currentDataframe = pd.DataFrame({'timeStamp': timestamps})
    
            # Add user information to each row
            for key, value in user_data.items():
                currentDataframe[key] = value
    
            df_list.append(currentDataframe)

    except json.JSONDecodeError:
        print(f"Error decoding {json_file}")
    except FileNotFoundError:
        print(f"Could not find {json_file}")
    except Exception as e:
        print(f"Error processing {json_file}: {str(e)}")

#print(len(df_list))
allDataDF = pd.concat(df_list, ignore_index=True)

# Save allDataDF for use in other notebooks
notebook_data_path = Path("../data/notebookData")
notebook_data_path.mkdir(parents=True, exist_ok=True)
allDataDF.to_parquet(notebook_data_path / "allDataDF.parquet")

# Summary table
summaryDF = pd.DataFrame(
    {'Value': [allDataDF.shape[0], allDataDF['userID'].nunique()]},
    index=pd.Index(['Number of marks', 'Unique participants'], name='Metric')
)
display(summaryDF)

,Value
Metric,
Number of marks,347
Unique participants,49


### 3. Personal Background ###

In [10]:
from IPython.display import HTML

def show_background_tables(dataDF):
    # group
    all_ages = dataDF.groupby('userID')['age'].first()
    all_gender = dataDF.groupby('userID')['gender'].first()
    all_musicaltraining = dataDF.groupby('userID')['musicaltraining'].first()
    all_experienceimprovising = dataDF.groupby('userID')['experienceimprovising'].first()

    # skip missing data
    all_ages = all_ages[all_ages != 0] # leave out blank form data
    all_ages = all_ages[all_ages < 100] # leave out apparently wrong data
    #all_familiarity = all_familiarity[all_familiarity != -99]
    #all_measuringdevice = all_measuringdevice[all_measuringdevice != -99]

    print("Number of participants: ", dataDF['userID'].nunique())

    # means and medians
    statsDF = pd.DataFrame({
        'Age': all_ages,
        'Musical training': all_musicaltraining,
        'Experience impro': all_experienceimprovising,
    }).agg(['count', 'mean', 'median', 'min', 'max']).T
    statsDF.columns = ['N', 'Mean', 'Median', 'Min', 'Max']
    statsDF.index.name = 'Variable'

    # gender categories
    genderDF = (
        all_gender.map({1: 'Diverse', 2: 'Male', 3: 'Female'})
        .value_counts()
        .reindex(['Diverse', 'Male', 'Female'], fill_value=0)
        .rename_axis('Gender')
        .to_frame('N')
    )
    genderDF['%'] = (genderDF['N'] / genderDF['N'].sum() * 100).round(1)

    # distribution of Likert ratings (1-5)
    likertDF = pd.DataFrame({
        'Musical training': all_musicaltraining.value_counts(),
        'Experience impro': all_experienceimprovising.value_counts(),
    }).reindex(range(1, 6), fill_value=0).T
    likertDF.columns.name = 'Rating'

    # show the three tables side by side
    tables = [
        statsDF.style.format({'N': '{:.0f}', 'Mean': '{:.2f}', 'Median': '{:.1f}', 'Min': '{:.0f}', 'Max': '{:.0f}'}).set_caption('Descriptive statistics'),
        genderDF.style.format({'%': '{:.1f}'}).set_caption('Gender'),
        likertDF.style.set_caption('Rating distribution'),
    ]
    display(HTML(
        '<div style="display: flex; gap: 2em; align-items: flex-start;">'
        + ''.join(f'<div>{t.to_html()}</div>' for t in tables)
        + '</div>'
    ))
    return statsDF, genderDF, likertDF

# Tables for the whole group
statsDF, genderDF, likertDF = show_background_tables(allDataDF)

Number of participants:  49


#### Participants 101 to 112 ####

In [11]:
# Same tables for the participants with IDs 101 to 112
group101to112DF = allDataDF[allDataDF['userID'].between(101, 112)]
stats101to112DF, gender101to112DF, likert101to112DF = show_background_tables(group101to112DF)

Number of participants:  12
